# Fix dataset paths

This notebook will try to change the built-in paths on a previously segmented dataset.
Useful if you want to move processing somewhere.

In [ ]:
# this cell is tagged with 'parameters'
# to view the tag, select the cell, then find the settings gear icon (right or left sidebar) and look for Cell Tags

# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

# dataset file to import
dset_path = '/path/to/dataset.h5'

# EXPERTS: Can specify par_file as a parameter if you want
par_file = '/path/to/pars.json'

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import ImageD11.columnfile
import ImageD11.sinograms.dataset

# Load data
## Dataset

In [ ]:
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

In [ ]:
ds.analysisroot = '/path/to/new/analysisroot'
ds.analysispath = None
ds.dsfile = None
ds.update_paths(force=True)

# also sort out the spatial files
# double-check if you're using e2dx/y or detectorh5
ds.e2dxfile = '/path/to/e2dx.edf'
ds.e2dyfile = '/path/to/e2dy.edf'
ds.detectorh5 = '/path/to/deth5.h5'

ds.save()

In [ ]:
# this should work assuming you already did segmentation and assemble+label and properties
!ls {ds.pksfile}

In [ ]:
if par_file is not None:
    # only change if ds has no parfile
    if not hasattr(ds, 'parfile') or ds.parfile is None:
        ds.parfile = par_file
        ds.save()

## Peaks

In [ ]:
# this is mostly to check whether it worked.

cf_4d = ds.get_cf_4d()
ds.update_colfile_pars(cf_4d)
if not os.path.exists(ds.col4dfile):
    # save the 4D peaks to file so we don't have to spatially correct them again
    ImageD11.columnfile.colfile_to_hdf(cf_4d, ds.col4dfile)